# RF-DETR Training — Insect Detection on Synthetic Trays

This notebook fine-tunes an RF-DETR model on the synthetic insect tray dataset generated by `generate_synthetic_trays_DATASET.ipynb`.

**Workflow:**
1. Install dependencies
2. Configure all settings (edit this section before every run)
3. Verify the dataset is correctly formatted
4. Train the model
5. Inspect predictions on test images
6. Export to ONNX for deployment

---
**Requirements before running:**
- Run `generate_synthetic_trays_DATASET.ipynb` → Cell 12 (generate trays) → Cell 16 (export COCO dataset) first.
- The `insect_dataset_coco/` folder must exist with `train/`, `valid/`, and `test/` sub-folders, each containing images and `_annotations.coco.json`.


---
## Cell 1 — Install Dependencies

Installs RF-DETR with training and logging extras from the local source folder.
Only needs to be run **once per environment** (skip if already installed).


In [ ]:
import subprocess, sys, os
from pathlib import Path

# Resolve the rf-detr source directory relative to this notebook's location.
# Works whether you launch Jupyter from the project root or anywhere else.
_nb_dir  = Path(os.path.abspath(''))
_rfdetr  = _nb_dir / 'rf-detr-1.7.1'

if not _rfdetr.exists():
    raise FileNotFoundError(
        f"Cannot find rf-detr source at {_rfdetr}.\n"
        "Make sure this notebook is in the same folder as rf-detr-1.7.1/."
    )

# Install with training + logging extras.
# The editable install (-e) means changes to the source are picked up immediately.
print(f"Installing from: {_rfdetr}")
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "-e", f"{_rfdetr}[train,loggers]",
])

print("\nInstallation complete.")
print("IMPORTANT: Restart the kernel now (Kernel → Restart), then run from Cell 2 onwards.")
print("You do NOT need to re-run this cell after restarting.")


---
## Cell 2 — Configuration

> **This is the main section to edit before every run.**
>
> All tuneable knobs are collected here so you never have to hunt through the rest of the notebook.


In [ ]:
# ============================================================
# SEPTEMBER RUN — RF-DETR, whole tray @1536
# ============================================================
# Regenerated dataset, trained at full resolution. What changed since August:
#
#   * GROUND TRUTH boxes are the insect, not the rotated canvas. The old boxes were
#     1.295x too large on a side (4/pi for a uniform angle), so ~40% of every box was
#     tray. Every AP number from before was measured against inflated targets.
#   * TRAYS are 3473x2769 = 10.85 px/mm, matching the real 3DTRAB trays exactly.
#     Previously 2644x2108 = 8.26 px/mm, so synthetic insects were 24% too small.
#   * TAXONOMY is 22 classes, not 23. Siphonaptera and Raphidioptera dropped;
#     Diplopoda -> Polyxenida, Chilopoda -> Lithobiomorpha + Scutigeromorpha.
#   * DATA is synthetic + real 3DTRAB mixed, group-disjoint on the real side, with
#     the real half oversampled to ~5% of the training signal.
#
# Evaluate on 3dtrab_coco_TESTSPLIT.json, NEVER the full 3DTRAB — roughly half of
# those photographs are in this training set.

from pathlib import Path


import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # 0=RF-DETR whole, 1=flatbug whole, 2=RF-DETR tiled, 3=flatbug tiled
# ─────────────────────────────────────────────────────────────────────────────
# PATHS  ← Change these if your folder layout is different
# ─────────────────────────────────────────────────────────────────────────────

# Root folder of this project (where this notebook lives).
BASE_DIR = Path(os.environ.get("MA_BASE", "/data/MA_Atul_Ares"))   # <-- Ares path; matches final_comparison

# COCO dataset produced by the export cell of generate_synthetic_trays_DATASET_server.ipynb, then mixed by mix_real_and_synthetic.ipynb.
# Must contain: train/_annotations.coco.json, valid/_annotations.coco.json,
#               test/_annotations.coco.json  (+ images in each sub-folder).
DATASET_DIR = str(BASE_DIR / "insect_dataset_mixed")   # synthetic + real, mixed

# Where checkpoints and logs will be saved.  A new sub-folder is created per run.
OUTPUT_DIR = str(BASE_DIR / "sept_run" / "rfdetr_whole_1536")

# ─────────────────────────────────────────────────────────────────────────────
# MODEL VARIANT  ← Choose one; see Cell 3 (markdown) for a comparison table
# ─────────────────────────────────────────────────────────────────────────────

# Options (in order of speed → accuracy):
#   RFDETRNano    384 px  30.5 M params  COCO AP50 67.6  — fastest, good for CPU/edge
#   RFDETRSmall   512 px  32.1 M params  COCO AP50 72.1
#   RFDETRMedium  576 px  33.7 M params  COCO AP50 73.6  ← recommended starting point
#   RFDETRLarge   704 px  33.9 M params  COCO AP50 75.1  — best accuracy, needs more VRAM
MODEL_VARIANT = "RFDETRLarge"

# ─────────────────────────────────────────────────────────────────────────────
# CLASSES  ← Must match what was used in the COCO export (Cell 16)
# ─────────────────────────────────────────────────────────────────────────────

# Single-class detection: all insects are labelled as "insect".
# Extend this list if you ever add per-order class labels.
CLASS_NAMES = [
    # Insecta — 17 orders.  Siphonaptera (6 specimens) and Raphidioptera (2) were
    # dropped from the taxonomy; their specimens are on the prune blocklist so they
    # never reached a tray.
    "Lepidoptera", "Psocodea", "Odonata", "Zoraptera",
    "Phasmatodea", "Zygentoma", "Thysanoptera", "Strepsiptera",
    "Archaeognatha", "Dermaptera",
    "Diptera", "Coleoptera", "Hymenoptera", "Hemiptera",
    "Orthoptera", "Trichoptera", "Neuroptera",
    # Arachnida — 2 orders
    "Araneae", "Opiliones",
    # Myriapods at ORDER level.  Previously the class-level labels "Diplopoda" and
    # "Chilopoda", each covering morphologically unrelated orders — which is why both
    # scored badly (0.19 and 0.34).
    "Polyxenida", "Lithobiomorpha", "Scutigeromorpha",
]

# ─────────────────────────────────────────────────────────────────────────────
# TRAINING SCHEDULE
# ─────────────────────────────────────────────────────────────────────────────

# Total training epochs.  50 is a safe default for fine-tuning.
# Increase to 100–150 if mAP is still climbing at the end.
EPOCHS = 100

# Images per GPU per forward pass.  Reduce to 2 if you run out of VRAM.
# Set to "auto" to let RF-DETR probe VRAM and choose automatically.
BATCH_SIZE = 4   # 1536 is 3.7x the pixels of 800; drop to 2 (and GRAD_ACCUM to 8) on OOM

# Gradient accumulation: effective batch = BATCH_SIZE × GRAD_ACCUM_STEPS.
# With batch_size=4 and grad_accum=4 the effective batch is 16 (the recommended default).
# Increase grad_accum if you lower batch_size due to VRAM constraints.
GRAD_ACCUM_STEPS = 4

# ─────────────────────────────────────────────────────────────────────────────
# LEARNING RATE
# ─────────────────────────────────────────────────────────────────────────────

# Decoder learning rate.  1e-4 is the RF-DETR default and works well for fine-tuning.
LR = 1e-4

# Encoder (DINOv2 backbone) learning rate.  Should be slightly higher than LR
# so the backbone adapts faster than the decoder heads.
LR_ENCODER = 1.5e-4

# Epoch at which the learning rate is multiplied by 0.1 (cosine step-down).
# Set to EPOCHS so the LR never drops, or to e.g. 40 for a 50-epoch run.
LR_DROP = 50  # no LR drop by default; change to e.g. 40

# Warm-up epochs at the start of training (ramps LR from 0 → LR linearly).
# 0 = disabled.  Try 5 if training is unstable in the first few epochs.
WARMUP_EPOCHS = 5

# ─────────────────────────────────────────────────────────────────────────────
# REGULARISATION
# ─────────────────────────────────────────────────────────────────────────────

# L2 weight decay applied to all parameters except biases and norms.
WEIGHT_DECAY = 1e-4

# ─────────────────────────────────────────────────────────────────────────────
# DATA AUGMENTATION
# ─────────────────────────────────────────────────────────────────────────────

# Multi-scale training: randomly resizes images between 0.5× and 1.5× during training.
# Strongly recommended — keeps True unless you need strict reproducibility.
MULTI_SCALE = True

# ─────────────────────────────────────────────────────────────────────────────
# CHECKPOINTING & EVALUATION
# ─────────────────────────────────────────────────────────────────────────────

# Save a checkpoint every N epochs (in addition to the best-mAP checkpoint).
CHECKPOINT_INTERVAL = 10

# Evaluate on the validation set every N epochs.  1 = every epoch (default).
EVAL_INTERVAL = 1

# Also run evaluation on the TEST split at the end of training.
# Produces a separate mAP score for your held-out test set.
RUN_TEST = True

# ─────────────────────────────────────────────────────────────────────────────
# EARLY STOPPING  (optional)
# ─────────────────────────────────────────────────────────────────────────────

# Stop training automatically if val mAP does not improve.
# Set to False if you want to train for the full EPOCHS regardless.
EARLY_STOPPING = True

# How many epochs without improvement before stopping.
EARLY_STOPPING_PATIENCE = 20

# Minimum improvement in mAP that counts as "better".
EARLY_STOPPING_MIN_DELTA = 0.001

# ─────────────────────────────────────────────────────────────────────────────
# EMA  (Exponential Moving Average — recommended ON)
# ─────────────────────────────────────────────────────────────────────────────

# EMA maintains a shadow copy of model weights and produces more stable predictions.
# Uses ~30 % extra VRAM.  Set to False only if you are tight on memory.
USE_EMA = False


RESOLUTION = 1536
# Trays are 3473 px wide, so this feeds them at 0.44x. A real beetle lands at ~58 px
# and a synthetic one at ~31 px; at the old 800 they were 30 and 16 px, i.e. below
# COCO's 32 px "small object" line where detectors struggle most. 1536 is a multiple
# of 32 as RF-DETR requires. Raise to 2048 (41/77 px) if VRAM allows.
# ─────────────────────────────────────────────────────────────────────────────
# LOGGING
# ─────────────────────────────────────────────────────────────────────────────

# TensorBoard: writes loss/mAP curves to OUTPUT_DIR.  Launch with:
#   tensorboard --logdir output/
TENSORBOARD = True

# Weights & Biases: set to True and call `wandb login` in the terminal first.
WANDB = False

# ─────────────────────────────────────────────────────────────────────────────
# DATALOADER
# ─────────────────────────────────────────────────────────────────────────────

# CPU worker threads for the dataloader.  Increase to 4–8 if training is I/O-bound.
# Set to 0 to load data in the main process (useful for debugging).
# IMPORTANT for multi-GPU: set to 0 when using DDP (ddp_notebook).
# Spawned DDP processes + spawned dataloader workers deadlock each other.
# With 311 small tray images the dataset fits easily; 0 workers is fast enough.
# You can raise this to 2-4 only when training on a single GPU (NUM_GPUS=1).
# 0 was needed to avoid spawn-DDP deadlocks in Jupyter.
# For torchrun (Cell 6b), use 2 — safe and prevents COCO-loading deadlock.
NUM_WORKERS = 8

# ─────────────────────────────────────────────────────────────────────────────
# RESUME  (leave None to start fresh)
# ─────────────────────────────────────────────────────────────────────────────

# Path to a checkpoint file to resume training from.
# Example: RESUME_FROM = "output/checkpoint_best_total.pth"
# Leave as None to always start from the ImageNet/COCO pretrained weights.
RESUME_FROM = None
# ─────────────────────────────────────────────────────────────────────────────
# INFERENCE SETTINGS  (used in Cell 7 — visual inspection)
# ─────────────────────────────────────────────────────────────────────────────

# Confidence threshold for displaying predictions.
# Boxes below this score are hidden.  0.3–0.5 is typical.
INFERENCE_THRESHOLD = 0.6

# How many test images to show in the inspection grid.
NUM_INSPECT_IMAGES = 6

# ─────────────────────────────────────────────────────────────────────────────
# PROGRESS BAR
# ─────────────────────────────────────────────────────────────────────────────

# Progress bar style during training.
#   "rich"  — coloured table with loss/mAP columns, best for Jupyter notebooks
#   "tqdm"  — classic tqdm bar (simpler, also works in terminals)
#   None    — no progress bar (silent; only epoch-level logs printed)
PROGRESS_BAR = "rich"

# ─────────────────────────────────────────────────────────────────────────────
# MULTI-GPU
# ─────────────────────────────────────────────────────────────────────────────

# Number of GPUs to use.  Set to 1 for single-GPU training.
# With 2 GPUs the global effective batch doubles:
#   effective batch = BATCH_SIZE x GRAD_ACCUM_STEPS x NUM_GPUS
# If still tight on VRAM per GPU, lower BATCH_SIZE to 2 and raise GRAD_ACCUM_STEPS to 8.
NUM_GPUS = 1

# DDP strategy for multi-GPU inside a Jupyter notebook.
# "ddp_notebook" uses spawn-based DDP — safe in notebooks (avoids fork/CUDA conflicts).
# Automatically switches to "auto" when NUM_GPUS = 1.
DDP_STRATEGY = "ddp_notebook" if NUM_GPUS > 1 else "auto"

# ─────────────────────────────────────────────────────────────────────────────
print("Configuration loaded.")
print(f"  Dataset : {DATASET_DIR}")
print(f"  Model   : {MODEL_VARIANT}")
print(f"  Epochs  : {EPOCHS}   |   Eff. batch: {BATCH_SIZE} x {GRAD_ACCUM_STEPS} x {NUM_GPUS} GPU(s) = {BATCH_SIZE * GRAD_ACCUM_STEPS * NUM_GPUS}")
print(f"  LR      : {LR}  (encoder: {LR_ENCODER})")
print(f"  GPUs    : {NUM_GPUS}  strategy: {DDP_STRATEGY}")
print(f"  Output  : {OUTPUT_DIR}")


---
## Cell 3 — Model Variant Reference

Use this table to pick `MODEL_VARIANT` in Cell 2.

| Variant | Resolution | Params | COCO AP50 | COCO AP50:95 | GPU latency | Notes |
|---|---|---|---|---|---|---|
| `RFDETRNano`   | 384 × 384 | 30.5 M | 67.6 | 48.4 | 2.3 ms | Fastest; good for CPU / edge / low VRAM |
| `RFDETRSmall`  | 512 × 512 | 32.1 M | 72.1 | 53.0 | 3.5 ms | Good balance of speed + accuracy |
| `RFDETRMedium` | 576 × 576 | 33.7 M | 73.6 | 54.7 | 4.4 ms | **Recommended starting point** |
| `RFDETRLarge`  | 704 × 704 | 33.9 M | 75.1 | 56.5 | 6.8 ms | Best accuracy; needs ≥ 16 GB VRAM for batch 4 |

> **Rule of thumb for tray images (large, high-resolution):**
> Start with `RFDETRMedium`. If you see many missed small insects, try `RFDETRLarge` for the larger receptive field.
> If VRAM is limited (< 8 GB), use `RFDETRSmall` with `batch_size=2, grad_accum_steps=8`.


---
## Cell 4 — Verify Dataset

Checks that the COCO dataset is correctly laid out before wasting time starting a training run.


In [ ]:
import json
from pathlib import Path

_ds = Path(DATASET_DIR)

print("Checking dataset structure ...")
ok = True

for split in ["train", "valid", "test"]:
    ann_path = _ds / split / "_annotations.coco.json"
    if not ann_path.exists():
        print(f"  [MISSING] {ann_path}")
        ok = False
        continue

    with open(ann_path) as _f:
        _ann = json.load(_f)

    n_images = len(_ann.get("images", []))
    n_anns   = len(_ann.get("annotations", []))
    n_cats   = len(_ann.get("categories", []))
    cat_names = [c["name"] for c in _ann.get("categories", [])]

    # Count images that actually exist on disk
    n_on_disk = sum(
        1 for img in _ann["images"]
        if (_ds / split / img["file_name"]).exists()
    )

    status = "OK" if n_on_disk == n_images else f"WARNING — only {n_on_disk}/{n_images} images found on disk"
    print(f"  [{status}] {split:6s}  {n_images:4d} images  {n_anns:6d} annotations  {n_cats} class(es): {cat_names}")
    if n_on_disk != n_images:
        ok = False

if ok:
    print("\nDataset looks good — ready to train.")
else:
    print("\nFix the issues above before proceeding.")
    print("Re-run Cell 16 of generate_synthetic_trays_DATASET.ipynb to rebuild the dataset.")


---
## Cell 5 — Initialise Model

Creates the RF-DETR model object and downloads the COCO pretrained backbone weights (first run only, ~130 MB).


In [ ]:
import sys
from pathlib import Path

# ── Ensure rfdetr is importable ─────────────────────────────────────────────
# If the pip install in Cell 1 was done in a previous session and the package
# is already on sys.path, this block is a no-op.  Otherwise it adds the source
# directory directly so the import works without reinstalling.
_src_path = str(Path(".").resolve() / "rf-detr-1.7.1" / "src")
if _src_path not in sys.path:
    sys.path.insert(0, _src_path)

try:
    from rfdetr import RFDETRNano, RFDETRSmall, RFDETRMedium, RFDETRLarge
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        "Cannot import rfdetr.  Run Cell 1 to install it, restart the kernel, "
        "then continue from Cell 2."
    ) from e

# Map the string name chosen in Cell 2 to the actual class.
_VARIANT_MAP = {
    "RFDETRNano":   RFDETRNano,
    "RFDETRSmall":  RFDETRSmall,
    "RFDETRMedium": RFDETRMedium,
    "RFDETRLarge":  RFDETRLarge,
}

if MODEL_VARIANT not in _VARIANT_MAP:
    raise ValueError(
        f"Unknown MODEL_VARIANT '{MODEL_VARIANT}'.  "
        f"Choose from: {list(_VARIANT_MAP)}"
    )

ModelClass = _VARIANT_MAP[MODEL_VARIANT]

# Instantiating with no arguments loads the COCO pretrained weights automatically.
# Pass pretrain_weights=None to start from a random initialisation instead.
model = ModelClass(resolution=RESOLUTION)

print(f"Model initialised: {MODEL_VARIANT}")
print(f"  Input resolution : {model.model_config.resolution} px")
print(f"  Encoder          : {model.model_config.encoder}")


---
## Cell 6 — Train

Starts the training loop.  All parameters come from the configuration in Cell 2.

**Checkpoints saved to `OUTPUT_DIR`:**
- `checkpoint_best_total.pth` — best validation mAP (use this for inference / export)
- `checkpoint_<epoch>.pth` — periodic snapshots every `CHECKPOINT_INTERVAL` epochs
- `checkpoint_last.pth` — most recent epoch (for resuming)

**Monitor training progress:**
```bash
tensorboard --logdir output/
```


In [ ]:
model.train(
    # ── Dataset ──────────────────────────────────────────────────────────────
    dataset_dir          = DATASET_DIR,      # root folder with train/valid/test sub-dirs
    output_dir           = OUTPUT_DIR,       # where checkpoints and logs are written

    # ── Classes ──────────────────────────────────────────────────────────────
    class_names          = CLASS_NAMES,      # must match category names in _annotations.coco.json

    # ── Schedule ─────────────────────────────────────────────────────────────
    epochs               = EPOCHS,
    batch_size           = BATCH_SIZE,       # per-GPU batch size
    grad_accum_steps     = GRAD_ACCUM_STEPS, # effective batch = batch_size × grad_accum_steps

    # ── Learning rate ─────────────────────────────────────────────────────────
    lr                   = LR,              # decoder head LR
    lr_encoder           = LR_ENCODER,     # DINOv2 backbone LR (slightly higher recommended)
    lr_drop              = LR_DROP,        # epoch at which LR is multiplied by 0.1
    warmup_epochs        = WARMUP_EPOCHS,  # linear warm-up from 0 → lr

    # ── Regularisation ───────────────────────────────────────────────────────
    weight_decay         = WEIGHT_DECAY,

    # ── Augmentation ─────────────────────────────────────────────────────────
    multi_scale          = MULTI_SCALE,    # random resize between 0.5× and 1.5×

    # ── EMA ──────────────────────────────────────────────────────────────────
    use_ema              = USE_EMA,        # exponential moving average of weights

    # ── Checkpointing ────────────────────────────────────────────────────────
    checkpoint_interval  = CHECKPOINT_INTERVAL,  # save every N epochs
    eval_interval        = EVAL_INTERVAL,        # evaluate val split every N epochs
    run_test             = RUN_TEST,             # evaluate test split at end of training

    # ── Early stopping ───────────────────────────────────────────────────────
    early_stopping            = EARLY_STOPPING,
    early_stopping_patience   = EARLY_STOPPING_PATIENCE,
    early_stopping_min_delta  = EARLY_STOPPING_MIN_DELTA,

    # ── Logging ──────────────────────────────────────────────────────────────
    # ── Progress bar ──────────────────────────────────────────────────────────
    progress_bar         = PROGRESS_BAR,  # "rich", "tqdm", or None

    tensorboard          = TENSORBOARD,
    wandb                = WANDB,

    # ── Dataloader ───────────────────────────────────────────────────────────
    num_workers          = NUM_WORKERS,

    # ── Resume (None = start fresh) ───────────────────────────────────────────

    # ── Multi-GPU ────────────────────────────────────────────────────────────
    devices              = NUM_GPUS,       # number of GPUs (set in Cell 2)
    strategy             = DDP_STRATEGY,   # spawn-based DDP, safe inside Jupyter

    resume               = RESUME_FROM,
)


## OPTION — Two-stage transfer: real flat-bug detection → synthetic

Run these **instead of** the single-stage `model.train(...)` cell above. RF-DETR starts from generic COCO, so this gives it the real-arthropod localisation prior flatbug already has. **Stage 1**: pretrain on the flat-bug data (converted YOLO→COCO, 1-class *detection*). **Stage 2**: fine-tune on the synthetic 22-class trays from the Stage-1 checkpoint — the class head auto-reinitialises 1→23. Stage-2 writes to `OUTPUT_DIR`, so Cell 15 + the metrics section work unchanged.


In [ ]:
# ============================================================
# TWO-STAGE config — real flat-bug detection pretrain -> synthetic
# ============================================================
# Stage 1 pretrains RF-DETR on the REAL flat-bug arthropod dataset (1-class
# 'insect' DETECTION) for a real-world localisation prior; Stage 2 fine-tunes on
# the mixed 22-class trays (the class head is re-initialised 1 -> 23).
#
# Prepare the flat-bug data first (server): s5cmd sync s3://flat-bug-data-stable/pre-pro/* ./pre-pro
#   (AWS keys, or Zenodo 10.5281/zenodo.14761446)  then  fb_prepare_data -i ./pre-pro -o ./fb_yolo
import sys
from pathlib import Path
_src = str(Path(".").resolve() / "rf-detr-1.7.1" / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)
from rfdetr import RFDETRNano, RFDETRSmall, RFDETRMedium, RFDETRLarge
ModelClass = {"RFDETRNano": RFDETRNano, "RFDETRSmall": RFDETRSmall,
              "RFDETRMedium": RFDETRMedium, "RFDETRLarge": RFDETRLarge}[MODEL_VARIANT]

FB_YOLO_YAML      = str(BASE_DIR / "fb_yolo" / "data.yaml")   # <-- prepared flat-bug YOLO data
FLATBUG_COCO_DIR  = str(BASE_DIR / "flatbug_coco")           # YOLO->COCO output (1-class 'insect')
STAGE1_OUTPUT_DIR = OUTPUT_DIR + "_stage1_flatbug"
STAGE1_EPOCHS     = 50                                       # real-data pretrain (Stage 2 uses EPOCHS)
assert Path(FB_YOLO_YAML).exists(), f"flat-bug YOLO data.yaml not found: {FB_YOLO_YAML} (run fb_prepare_data first)"
print(f"Stage 1: {FB_YOLO_YAML} -> {FLATBUG_COCO_DIR} ({STAGE1_EPOCHS} ep)  |  Stage 2: {DATASET_DIR} ({EPOCHS} ep)")


In [ ]:
# ── Convert the flat-bug YOLO data -> COCO (1-class) for RF-DETR ──
import json, shutil
from pathlib import Path
from PIL import Image

_IMG = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

def _load_yaml(p):
    try:
        import yaml
        return yaml.safe_load(open(p))
    except Exception:
        # minimal fallback parser for simple key: value / list yaml
        cfg = {}
        for ln in Path(p).read_text().splitlines():
            ln = ln.split("#", 1)[0].rstrip()
            if not ln or ":" not in ln:
                continue
            k, v = ln.split(":", 1); k = k.strip(); v = v.strip()
            if v.startswith("[") and v.endswith("]"):
                cfg[k] = [x.strip().strip("'\"") for x in v[1:-1].split(",") if x.strip()]
            elif v:
                cfg[k] = v.strip("'\"")
        return cfg

def _img2label(img_path):
    """Ultralytics rule: replace the last '/images/' with '/labels/', ext -> .txt."""
    p = Path(img_path)
    parts = list(p.parts)
    if "images" in parts:
        i = len(parts) - 1 - parts[::-1].index("images")
        parts[i] = "labels"
        return Path(*parts).with_suffix(".txt")
    return p.parent.parent / "labels" / (p.stem + ".txt")   # fallback: sibling labels/

def _resolve_images(root, spec):
    """spec may be a dir, a .txt list file, or a list of those."""
    specs = spec if isinstance(spec, list) else [spec]
    imgs = []
    for s in specs:
        sp = (root / s) if not Path(s).is_absolute() else Path(s)
        if sp.is_dir():
            imgs += [q for q in sp.rglob("*") if q.suffix.lower() in _IMG]
        elif sp.suffix.lower() == ".txt" and sp.exists():
            for ln in sp.read_text().splitlines():
                ln = ln.strip()
                if not ln:
                    continue
                q = Path(ln)
                if not q.is_absolute():
                    q = (root / q).resolve()
                imgs.append(q)
        elif sp.suffix.lower() in _IMG:
            imgs.append(sp)
    return sorted(set(imgs))

def _yolo_line_to_xyxy_norm(tokens):
    """Return (cls, x1, y1, x2, y2) NORMALISED, handling BOTH ultralytics label formats:
      * detection: `cls cx cy w h`                      (5 tokens)
      * segmentation: `cls x1 y1 x2 y2 x3 y3 ...`       (polygon; what convert_coco
        writes with use_segments=True, e.g. the flat-bug data) -> take the polygon's
        min/max extent as the box.
    Reading a polygon line as `cx cy w h` silently produces enormous wrong boxes."""
    v = [float(x) for x in tokens[1:]]
    c = int(float(tokens[0]))
    if len(v) == 4:
        cx, cy, w, h = v
        return c, cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2
    if len(v) >= 6 and len(v) % 2 == 0:
        xs, ys = v[0::2], v[1::2]
        return c, min(xs), min(ys), max(xs), max(ys)
    return None

def yolo_to_coco(data_yaml, out_dir, split_map=(("train", "train"), ("val", "valid"), ("test", "test")),
                 link=True):
    data_yaml = Path(data_yaml); out_dir = Path(out_dir)
    cfg = _load_yaml(data_yaml)
    root = Path(cfg.get("path", data_yaml.parent))
    if not root.is_absolute():
        root = (data_yaml.parent / root).resolve()
    names = cfg.get("names")
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names, key=lambda x: int(x))]
    if not names:
        names = ["insect"]
    categories = [{"id": i + 1, "name": n, "supercategory": "none"} for i, n in enumerate(names)]
    summary = {}
    for ysplit, csplit in split_map:
        if cfg.get(ysplit) is None:
            continue
        img_paths = _resolve_images(root, cfg[ysplit])
        if not img_paths:
            continue
        sdir = out_dir / csplit
        if sdir.exists():
            shutil.rmtree(sdir)
        sdir.mkdir(parents=True)
        images, annotations, ann_id = [], [], 1
        for img_id, ip in enumerate(img_paths, 1):
            try:
                with Image.open(ip) as im:
                    W, H = im.size
            except Exception:
                continue
            out_name = f"{img_id:07d}_{ip.name}"
            dst = sdir / out_name
            try:
                if link:
                    dst.symlink_to(ip.resolve())
                else:
                    shutil.copy(ip, dst)
            except Exception:
                shutil.copy(ip, dst)
            images.append({"id": img_id, "file_name": out_name, "width": W, "height": H})
            lp = _img2label(ip)
            if lp.exists():
                for ln in lp.read_text().splitlines():
                    t = ln.split()
                    if len(t) < 5:
                        continue
                    parsed = _yolo_line_to_xyxy_norm(t)
                    if parsed is None:
                        continue
                    c, nx1, ny1, nx2, ny2 = parsed
                    x = nx1 * W; y = ny1 * H; w = (nx2 - nx1) * W; h = (ny2 - ny1) * H
                    x = max(0, min(x, W - 1)); y = max(0, min(y, H - 1))
                    w = max(1, min(w, W - x)); h = max(1, min(h, H - y))
                    annotations.append({"id": ann_id, "image_id": img_id,
                                        "category_id": c + 1, "bbox": [x, y, w, h],
                                        "area": float(w * h), "iscrowd": 0})
                    ann_id += 1
        coco = {"info": {"description": f"YOLO->COCO ({csplit})"},
                "categories": categories, "images": images, "annotations": annotations}
        json.dump(coco, open(sdir / "_annotations.coco.json", "w"))
        summary[csplit] = (len(images), len(annotations))
    return names, summary

_names, _summary = yolo_to_coco(FB_YOLO_YAML, FLATBUG_COCO_DIR, link=True)
print("flat-bug COCO classes:", _names)
for _s, (_ni, _na) in _summary.items():
    print(f"  {_s:6s}: {_ni} images, {_na} annotations")


In [ ]:
# ── STAGE 1 — pretrain RF-DETR on real flat-bug data (1-class 'insect' detection) ──
m1 = ModelClass(resolution=RESOLUTION)
m1.train(
    dataset_dir=FLATBUG_COCO_DIR, output_dir=STAGE1_OUTPUT_DIR, class_names=["insect"],
    epochs=STAGE1_EPOCHS, batch_size=BATCH_SIZE, grad_accum_steps=GRAD_ACCUM_STEPS,
    lr=LR, lr_encoder=LR_ENCODER, lr_drop=STAGE1_EPOCHS, warmup_epochs=WARMUP_EPOCHS,
    weight_decay=WEIGHT_DECAY, multi_scale=MULTI_SCALE, use_ema=USE_EMA,
    checkpoint_interval=CHECKPOINT_INTERVAL, eval_interval=EVAL_INTERVAL, run_test=RUN_TEST,
    early_stopping=EARLY_STOPPING, early_stopping_patience=EARLY_STOPPING_PATIENCE,
    early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA, progress_bar=PROGRESS_BAR,
    tensorboard=TENSORBOARD, wandb=WANDB, num_workers=NUM_WORKERS,
    devices=NUM_GPUS, strategy=DDP_STRATEGY, resume=None,
)
STAGE1_CKPT = str(Path(STAGE1_OUTPUT_DIR) / "checkpoint_best_total.pth")
assert Path(STAGE1_CKPT).exists(), f"Stage-1 checkpoint not found: {STAGE1_CKPT}"
print(f"\nStage-1 checkpoint (real-arthropod localiser): {STAGE1_CKPT}")


In [ ]:
# ── STAGE 2 — fine-tune on synthetic 22-class, starting from the Stage-1 prior ──
# pretrain_weights loads Stage-1 backbone/encoder/decoder; RF-DETR auto-detects 23
# classes from the synthetic dataset and re-initialises the class head (1 -> 23).
model = ModelClass(resolution=RESOLUTION, pretrain_weights=STAGE1_CKPT)
model.train(
    dataset_dir=DATASET_DIR, output_dir=OUTPUT_DIR, class_names=CLASS_NAMES,
    epochs=EPOCHS, batch_size=BATCH_SIZE, grad_accum_steps=GRAD_ACCUM_STEPS,
    lr=LR, lr_encoder=LR_ENCODER, lr_drop=LR_DROP, warmup_epochs=WARMUP_EPOCHS,
    weight_decay=WEIGHT_DECAY, multi_scale=MULTI_SCALE, use_ema=USE_EMA,
    checkpoint_interval=CHECKPOINT_INTERVAL, eval_interval=EVAL_INTERVAL, run_test=RUN_TEST,
    early_stopping=EARLY_STOPPING, early_stopping_patience=EARLY_STOPPING_PATIENCE,
    early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA, progress_bar=PROGRESS_BAR,
    tensorboard=TENSORBOARD, wandb=WANDB, num_workers=NUM_WORKERS,
    devices=NUM_GPUS, strategy=DDP_STRATEGY,
    resume=None,   # fresh Stage-2 run (the prior comes from pretrain_weights, not resume)
)
print(f"\nStage-2 (final) checkpoints in: {OUTPUT_DIR}")
print("Load the best checkpoint below (Cell 15) and run the metrics section as usual.")


---
## Cell 6b — Multi-GPU Training (4× Titan X via torchrun)

Use this cell **instead of Cell 6** when training on multiple GPUs.
Jupyter's DDP spawner deadlocks with NCCL — this cell writes a small runner script
from the config above and launches it with `torchrun`, which uses proper DDP.

**All settings still come from Cell 2.** Change them there, then run this cell.


In [ ]:
import json, subprocess, sys, textwrap
from pathlib import Path

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1 — How many GPUs to use  (change this; everything else comes from Cell 2)
# ─────────────────────────────────────────────────────────────────────────────
NUM_GPUS_TORCHRUN = 4   # ← set to 1, 2, 3, or 4

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2 — Write config from Cell 2 variables to disk
# ─────────────────────────────────────────────────────────────────────────────
_nb_dir = Path('.').resolve()
_cfg = {
    "dataset_dir":              str(Path(DATASET_DIR).resolve()),
    "output_dir":               str(Path(OUTPUT_DIR).resolve()),
    "model_variant":            MODEL_VARIANT,
    "gradient_checkpointing":   True,
    "class_names":              CLASS_NAMES,
    "epochs":                   EPOCHS,
    "batch_size":               BATCH_SIZE,
    "grad_accum_steps":         GRAD_ACCUM_STEPS,
    "lr":                       LR,
    "lr_encoder":               LR_ENCODER,
    "lr_drop":                  LR_DROP,
    "warmup_epochs":            WARMUP_EPOCHS,
    "weight_decay":             WEIGHT_DECAY,
    "multi_scale":              MULTI_SCALE,
    "use_ema":                  USE_EMA,
    "checkpoint_interval":      CHECKPOINT_INTERVAL,
    "eval_interval":            EVAL_INTERVAL,
    "run_test":                 RUN_TEST,
    "early_stopping":           EARLY_STOPPING,
    "early_stopping_patience":  EARLY_STOPPING_PATIENCE,
    "early_stopping_min_delta": EARLY_STOPPING_MIN_DELTA,
    "progress_bar":             PROGRESS_BAR,
    "tensorboard":              TENSORBOARD,
    "wandb":                    WANDB,
    "num_workers":              NUM_WORKERS,
    "num_gpus":                 NUM_GPUS_TORCHRUN,
    "resume":                   RESUME_FROM,
}
_cfg_path = _nb_dir / 'train_config.json'
with open(_cfg_path, 'w') as _f:
    json.dump(_cfg, _f, indent=2)
print(f'Config written → {_cfg_path}')

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3 — Write the runner script inline (no external file needed)
# ─────────────────────────────────────────────────────────────────────────────
_runner_path = _nb_dir / '_rfdetr_runner.py'
_runner_src = textwrap.dedent(f"""
    import json, sys, math
    from pathlib import Path

    _src = str(Path('{_nb_dir}') / 'rf-detr-1.7.1' / 'src')
    if _src not in sys.path:
        sys.path.insert(0, _src)

    from rfdetr import RFDETRNano, RFDETRSmall, RFDETRMedium, RFDETRLarge

    _VARIANTS = {{
        'RFDETRNano':   RFDETRNano,
        'RFDETRSmall':  RFDETRSmall,
        'RFDETRMedium': RFDETRMedium,
        'RFDETRLarge':  RFDETRLarge,
    }}

    with open('{_cfg_path}') as f:
        cfg = json.load(f)

    # ── Pre-compute total optimizer steps from the COCO annotation count ──────
    # RF-DETR's LR scheduler calls trainer.estimated_stepping_batches, which
    # forces PTL to load the full train dataloader across all DDP ranks
    # simultaneously — this causes a collective-sync hang on 4 GPUs.
    # Patching it here means PTL never needs to touch the dataloader for estimation.
    _ann_file = Path(cfg['dataset_dir']) / 'train' / '_annotations.coco.json'
    with open(_ann_file) as _f:
        _n_train = len(json.load(_f)['images'])
    _bs   = cfg['batch_size']
    _ga   = cfg['grad_accum_steps']
    _gpus = cfg['num_gpus']
    _pad_unit        = _bs * _ga * _gpus
    _padded          = math.ceil(_n_train / _pad_unit) * _pad_unit
    _steps_per_epoch = (_padded // _gpus) // _bs // _ga
    _total_steps     = max(1, _steps_per_epoch) * cfg['epochs']
    print(f'[runner] train images: {{_n_train}}, steps/epoch: {{_steps_per_epoch}}, total steps: {{_total_steps}}')

    import pytorch_lightning as pl
    def _esb_patched(self):
        return _total_steps
    pl.Trainer.estimated_stepping_batches = property(_esb_patched)

    # ─────────────────────────────────────────────────────────────────────────

    model = _VARIANTS[cfg['model_variant']](
        gradient_checkpointing=cfg.get('gradient_checkpointing', True)
    )
    model.train(
        dataset_dir=cfg['dataset_dir'],
        output_dir=cfg['output_dir'],
        class_names=cfg['class_names'],
        epochs=cfg['epochs'],
        batch_size=cfg['batch_size'],
        grad_accum_steps=cfg['grad_accum_steps'],
        lr=cfg['lr'],
        lr_encoder=cfg['lr_encoder'],
        lr_drop=cfg['lr_drop'],
        warmup_epochs=cfg['warmup_epochs'],
        weight_decay=cfg['weight_decay'],
        multi_scale=cfg['multi_scale'],
        use_ema=cfg['use_ema'],
        checkpoint_interval=cfg['checkpoint_interval'],
        eval_interval=cfg['eval_interval'],
        run_test=cfg['run_test'],
        early_stopping=cfg['early_stopping'],
        early_stopping_patience=cfg['early_stopping_patience'],
        early_stopping_min_delta=cfg['early_stopping_min_delta'],
        progress_bar=cfg.get('progress_bar', 'rich'),
        tensorboard=cfg['tensorboard'],
        wandb=cfg['wandb'],
        num_workers=2,
        devices=cfg['num_gpus'],
        strategy='ddp',
        resume=cfg.get('resume') or None,
    )
""")
with open(_runner_path, 'w') as _f:
    _f.write(_runner_src)
print(f'Runner script written → {_runner_path}')

# ─────────────────────────────────────────────────────────────────────────────
# ─────────────────────────────────────────────────────────────────────────────
# STEP 4 — Launch training  (blocks until training finishes)
# Uses sys.executable so the rembg_gpu_env Python is used,
# NOT the system Python that the `torchrun` binary defaults to.
# ─────────────────────────────────────────────────────────────────────────────
_cmd = [
    sys.executable, '-m', 'torch.distributed.run',
    f'--nproc_per_node={NUM_GPUS_TORCHRUN}',
    str(_runner_path),
]
print(f'\nLaunching: {" ".join(_cmd)}')
print('=' * 60)

_proc = subprocess.Popen(
    _cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for _line in _proc.stdout:
    print(_line, end='', flush=True)
_proc.wait()

if _proc.returncode != 0:
    raise RuntimeError(f'Training failed with exit code {_proc.returncode}')
print('=' * 60)
print('Training complete. Checkpoints saved to:', OUTPUT_DIR)


---
## Cell 7 — Load Best Checkpoint

After training, load the best-mAP checkpoint for inference and export.

> **Note:** RF-DETR automatically syncs the best weights back into `model` when `train()` finishes,
> so this cell is only needed if you re-open the notebook **without** re-running the training cell.


In [ ]:
from rfdetr import RFDETR
from pathlib import Path

# Path to the best checkpoint saved during training.
# RF-DETR names it "checkpoint_best_total.pth" by default.
BEST_CHECKPOINT = Path(OUTPUT_DIR) / "checkpoint_best_total.pth"

if BEST_CHECKPOINT.exists():
    # from_checkpoint automatically detects the model variant from the file.
    model = RFDETR.from_checkpoint(str(BEST_CHECKPOINT), resolution=RESOLUTION)  # from_checkpoint does NOT restore resolution (falls back to the 704 default)
    print(f"Loaded checkpoint: {BEST_CHECKPOINT}")
else:
    # Training has already loaded the best weights into `model` — nothing to do.
    print(f"Checkpoint not found at {BEST_CHECKPOINT}.")
    print("If training just completed, the best weights are already loaded into `model`.")


---
## Cell 8 — Visual Inspection on Test Images

Runs inference on a sample of test images and displays predictions alongside ground-truth boxes.
Gives a quick qualitative feel for model quality before looking at mAP numbers.


In [ ]:
import random
import json
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from PIL import Image

# ── Load test annotations ─────────────────────────────────────────────────────
_test_ann_path = Path(DATASET_DIR) / "test" / "_annotations.coco.json"
with open(_test_ann_path) as _f:
    _test_coco = json.load(_f)

_id_to_img = {img["id"]: img for img in _test_coco["images"]}
_id_to_cat = {cat["id"]: cat["name"] for cat in _test_coco["categories"]}

# GT lookup: image_id → list of (bbox, class_name)
_gt_boxes = {}
for ann in _test_coco["annotations"]:
    _gt_boxes.setdefault(ann["image_id"], []).append(
        (ann["bbox"], _id_to_cat.get(ann["category_id"], "?"))
    )

# Sample a random subset of test images
_all_img_ids = list(_id_to_img.keys())
random.seed(42)
_sample_ids = random.sample(_all_img_ids, min(NUM_INSPECT_IMAGES, len(_all_img_ids)))

# ── Run inference and plot ────────────────────────────────────────────────────
_ncols = 2
_nrows = (len(_sample_ids) + _ncols - 1) // _ncols
fig, axes = plt.subplots(_nrows, _ncols, figsize=(14, 6 * _nrows))
axes = np.array(axes).flatten()

_all_summaries = []

for _ax, _img_id in zip(axes, _sample_ids):
    _img_meta  = _id_to_img[_img_id]
    _img_path  = Path(DATASET_DIR) / "test" / _img_meta["file_name"]
    _pil_image = Image.open(_img_path).convert("RGB")

    _detections = model.predict(_pil_image, threshold=INFERENCE_THRESHOLD)
    _ax.imshow(_pil_image)

    # ── Ground-truth boxes: labelled G1, G2, G3 ... ──────────────────────────
    _gt_legend = []
    for _gi, ((_x, _y, _w, _h), _gt_cls) in enumerate(_gt_boxes.get(_img_id, []), start=1):
        _lbl = f"G{_gi}"
        _ax.add_patch(mpatches.Rectangle(
            (_x, _y), _w, _h, linewidth=1, edgecolor="lime", facecolor="none"
        ))
        _ax.text(_x + 2, _y + 10, _lbl, color="lime", fontsize=5, fontweight="bold",
                 verticalalignment="top",
                 bbox=dict(facecolor="black", alpha=0.55, pad=1, linewidth=0))
        _gt_legend.append((_lbl, _gt_cls, None))

    # ── Predicted boxes: labelled P1, P2, P3 ... ─────────────────────────────
    _pred_legend = []
    if len(_detections) > 0:
        _class_names_arr = _detections.data.get("class_name", [])
        for _pi, ((_x1, _y1, _x2, _y2), _conf) in enumerate(
                zip(_detections.xyxy, _detections.confidence), start=1):
            _bw, _bh = _x2 - _x1, _y2 - _y1
            _cls_lbl = _class_names_arr[_pi - 1] if (_pi - 1) < len(_class_names_arr) else "?"
            _lbl = f"P{_pi}"
            _ax.add_patch(mpatches.Rectangle(
                (_x1, _y1), _bw, _bh, linewidth=1, edgecolor="red", facecolor="none"
            ))
            _ax.text(_x2 - 2, _y2 - 2, _lbl, color="red", fontsize=5, fontweight="bold",
                     horizontalalignment="right", verticalalignment="bottom",
                     bbox=dict(facecolor="black", alpha=0.55, pad=1, linewidth=0))
            _pred_legend.append((_lbl, _cls_lbl, _conf))

    _ax.set_title(
        f"{_img_meta['file_name']}  |  GT: {len(_gt_legend)}  Pred: {len(_pred_legend)}",
        fontsize=8
    )
    _ax.axis("off")

    _all_summaries.append({
        "file":        _img_meta["file_name"],
        "gt_legend":   _gt_legend,
        "pred_legend": _pred_legend,
    })

# Hide unused panels
for _ax in axes[len(_sample_ids):]:
    _ax.set_visible(False)

fig.legend(
    handles=[
        mpatches.Patch(edgecolor="lime", facecolor="none", label="Ground truth  (G1, G2, ...)"),
        mpatches.Patch(edgecolor="red",  facecolor="none",
                       label=f"Prediction (threshold={INFERENCE_THRESHOLD})  (P1, P2, ...)"),
    ],
    loc="upper right", fontsize=9
)
plt.tight_layout()

# Save grid to disk
_grid_path = Path(OUTPUT_DIR) / "test_inference_grid.png"
fig.savefig(_grid_path, dpi=150, bbox_inches="tight")
print(f"Saved → {_grid_path}")

plt.show()

# ── Per-image class key ───────────────────────────────────────────────────────
for _s in _all_summaries:
    print(f"\n{'═'*60}")
    print(f"  {_s['file']}")
    print(f"{'─'*60}")
    print(f"  {'ID':<6} {'Class':<22}  Conf")
    print(f"  {'─'*45}")
    for _lbl, _cls, _conf in _s["gt_legend"]:
        print(f"  {_lbl:<6} {_cls:<22}  (ground truth)")
    for _lbl, _cls, _conf in _s["pred_legend"]:
        print(f"  {_lbl:<6} {_cls:<22}  {_conf:.3f}")
print(f"{'═'*60}")


---
## Cell 9 — mAP Summary

Prints a concise summary of the best validation and test mAP from the training log saved in `OUTPUT_DIR`.

> If training was run with `run_test=True`, the test-set mAP is printed here too.


In [ ]:
import json
from pathlib import Path

# RF-DETR writes a summary JSON at the end of training.
# The exact filename may vary; look for any *_summary* or *log* JSON in OUTPUT_DIR.
_log_files = sorted(Path(OUTPUT_DIR).glob("*.json"))

if not _log_files:
    print("No JSON log files found in", OUTPUT_DIR)
    print("Check TensorBoard or the training stdout for mAP values.")
else:
    for _lf in _log_files:
        print(f"\n=== {_lf.name} ===")
        with open(_lf) as _f:
            _data = json.load(_f)
        if isinstance(_data, list):
            # Per-epoch log: print last entry
            _last = _data[-1]
            for _k, _v in _last.items():
                print(f"  {_k:40s}: {_v}")
        elif isinstance(_data, dict):
            for _k, _v in _data.items():
                print(f"  {_k:40s}: {_v}")


## Full evaluation metrics — confusion matrix, PR/F1 curves, results (flatbug-style)

Runs the trained model (loaded above) on the eval split, matches predictions to the COCO ground truth, and writes the same diagnostic files Ultralytics/flatbug produce — `confusion_matrix(_normalized).png`, `PR_curve/P_curve/R_curve/F1_curve.png`, `labels.jpg`, `results.png` (from RF-DETR's `metrics.csv`), `per_class_metrics.csv`, and `val_batch_*` previews — into `OUTPUT_DIR/metrics_<split>/`.


In [ ]:
# ============================================================
# METRICS TOOLKIT (self-contained; flatbug/Ultralytics-style plots)
# box_iou, match_image, compute_metrics, confusion_matrix,
# plot_confusion, plot_curves, plot_labels, plot_results
# ============================================================
import numpy as np, json, math
from pathlib import Path
import matplotlib.pyplot as plt

# ── geometry ──────────────────────────────────────────────────────────────────
def box_iou(a, b):
    """IoU matrix between xyxy boxes a:(N,4) b:(M,4)."""
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), np.float32)
    area_a = (a[:, 2] - a[:, 0]).clip(0) * (a[:, 3] - a[:, 1]).clip(0)
    area_b = (b[:, 2] - b[:, 0]).clip(0) * (b[:, 3] - b[:, 1]).clip(0)
    lt = np.maximum(a[:, None, :2], b[None, :, :2])
    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.clip(rb - lt, 0, None)
    inter = wh[..., 0] * wh[..., 1]
    union = area_a[:, None] + area_b[None, :] - inter
    return inter / np.clip(union, 1e-9, None)

# ── per-image matching (class-aware, greedy by IoU) → tp over IoU thresholds ──
def match_image(iou, pred_cls, gt_cls, thresholds):
    """Return tp (Np, T) boolean: pred i is a TP at threshold t."""
    Np = len(pred_cls); T = len(thresholds)
    tp = np.zeros((Np, T), bool)
    if iou.size == 0 or len(gt_cls) == 0:
        return tp
    same = pred_cls[:, None] == gt_cls[None, :]
    for ti, thr in enumerate(thresholds):
        ci, gi = np.where((iou >= thr) & same)
        if not len(ci):
            continue
        order = np.argsort(-iou[ci, gi])
        ci, gi = ci[order], gi[order]
        used_g, used_p = set(), set()
        for c, g in zip(ci, gi):
            if c in used_p or g in used_g:
                continue
            used_p.add(c); used_g.add(g); tp[c, ti] = True
    return tp

# ── AP + curves (COCO-style 101-pt interp; curves vs confidence) ──────────────
# NumPy 2.0 removed np.trapz in favour of np.trapezoid. The training envs run
# different NumPy majors (flatbug torch 2.13 / rfdetr torch 2.9), so bind once
# and use _TRAPZ rather than depending on which one is installed.
_TRAPZ = getattr(np, "trapezoid", None) or np.trapz
def _ap(recall, precision):
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([1.0], precision, [0.0]))
    mpre = np.flip(np.maximum.accumulate(np.flip(mpre)))
    x = np.linspace(0, 1, 101)
    return float(_TRAPZ(np.interp(x, mrec, mpre), x))

def compute_metrics(tp, conf, pred_cls, target_cls, nc, thresholds):
    """tp:(N,T). Returns dict of arrays + curves (Ultralytics layout)."""
    order = np.argsort(-conf)
    tp, conf, pred_cls = tp[order], conf[order], pred_cls[order]
    px = np.linspace(0, 1, 1000)
    T = tp.shape[1]
    ap = np.zeros((nc, T)); p_curve = np.zeros((nc, 1000)); r_curve = np.zeros((nc, 1000))
    f1_curve = np.zeros((nc, 1000)); prec_at = np.zeros((nc, 1000))  # precision vs recall(px)
    ap50 = np.zeros(nc); ap5095 = np.zeros(nc)
    n_gt = np.bincount(target_cls.astype(int), minlength=nc)
    n_p = np.bincount(pred_cls.astype(int), minlength=nc)
    for c in range(nc):
        i = pred_cls == c
        if n_gt[c] == 0 and i.sum() == 0:
            continue
        if i.sum() == 0:
            continue
        fpc = (1 - tp[i]).cumsum(0)
        tpc = tp[i].cumsum(0)
        recall = tpc / (n_gt[c] + 1e-16)
        precision = tpc / (tpc + fpc + 1e-16)
        # curves vs confidence threshold px (conf of this class's preds, desc)
        cc = conf[i]
        r_curve[c] = np.interp(-px, -cc, recall[:, 0], left=0)
        p_curve[c] = np.interp(-px, -cc, precision[:, 0], left=1)
        f1_curve[c] = 2 * p_curve[c] * r_curve[c] / (p_curve[c] + r_curve[c] + 1e-16)
        # precision vs recall (for PR_curve), at IoU 0.5 (threshold index 0)
        prec_at[c] = np.interp(px, recall[:, 0], precision[:, 0], left=1)
        for ti in range(T):
            ap[c, ti] = _ap(recall[:, ti], precision[:, ti])
        ap50[c] = ap[c, 0]; ap5095[c] = ap[c].mean()
    return dict(ap=ap, ap50=ap50, ap5095=ap5095, p_curve=p_curve, r_curve=r_curve,
                f1_curve=f1_curve, prec_at=prec_at, px=px, n_gt=n_gt, n_p=n_p)

# ── confusion matrix (fixed conf+iou, class-agnostic assignment) ──────────────
def confusion_matrix(images, nc, conf_thres=0.25, iou_thres=0.45):
    """images: list of dicts with pred_xyxy,pred_conf,pred_cls,gt_xyxy,gt_cls.
    matrix[pred, true], last row/col = background (FP row / FN col)."""
    cm = np.zeros((nc + 1, nc + 1), dtype=np.int64)
    for im in images:
        pc, pcf, pcl = im["pred_xyxy"], im["pred_conf"], im["pred_cls"]
        keep = pcf >= conf_thres
        pc, pcl = pc[keep], pcl[keep]
        gc, gcl = im["gt_xyxy"], im["gt_cls"]
        if len(gcl) == 0:
            for c in pcl:
                cm[int(c), nc] += 1            # predicted, no GT → FP (background col)
            continue
        if len(pcl) == 0:
            for c in gcl:
                cm[nc, int(c)] += 1            # GT missed → FN (background row)
            continue
        iou = box_iou(pc, gc)
        ci, gi = np.where(iou >= iou_thres)
        matched_p, matched_g = set(), set()
        if len(ci):
            order = np.argsort(-iou[ci, gi])
            for c, g in zip(ci[order], gi[order]):
                if c in matched_p or g in matched_g:
                    continue
                matched_p.add(c); matched_g.add(g)
                cm[int(pcl[c]), int(gcl[g])] += 1
        for c in range(len(pcl)):
            if c not in matched_p:
                cm[int(pcl[c]), nc] += 1        # FP
        for g in range(len(gcl)):
            if g not in matched_g:
                cm[nc, int(gcl[g])] += 1        # FN
    return cm

# ── plots ─────────────────────────────────────────────────────────────────────
def plot_confusion(cm, names, save, normalize=False):
    m = cm.astype(float)
    if normalize:
        m = m / (m.sum(0, keepdims=True) + 1e-9)   # column-normalize (per true class)
    labels = list(names) + ["background"]
    n = len(labels)
    fig, ax = plt.subplots(figsize=(max(9, n * 0.55), max(8, n * 0.5)))
    im = ax.imshow(m, cmap="Blues", vmin=0, vmax=(1.0 if normalize else None))
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(labels, rotation=90, fontsize=7)
    ax.set_yticklabels(labels, fontsize=7)
    ax.set_xlabel("True"); ax.set_ylabel("Predicted")
    ax.set_title("Confusion Matrix" + (" (normalized)" if normalize else ""))
    thr = (m.max() or 1) * 0.5
    if n <= 30:
        for i in range(n):
            for j in range(n):
                v = m[i, j]
                if v > 0:
                    ax.text(j, i, f"{v:.2f}" if normalize else f"{int(v)}",
                            ha="center", va="center", fontsize=6,
                            color="white" if v > thr else "black")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout(); fig.savefig(save, dpi=200); plt.close(fig)

def _curve(x, ys, names, xlabel, ylabel, title, save, mean_label):
    fig, ax = plt.subplots(figsize=(9, 6))
    ys = np.asarray(ys)
    for i, y in enumerate(ys):
        ax.plot(x, y, linewidth=1, label=names[i] if len(names) <= 25 else None)
    ax.plot(x, ys.mean(0), linewidth=3, color="navy", label=mean_label)
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.01); ax.grid(alpha=0.3)
    if len(names) <= 25:
        ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7)
    fig.tight_layout(); fig.savefig(save, dpi=200); plt.close(fig)

def plot_curves(M, names, outdir):
    px = M["px"]
    _curve(px, M["prec_at"], names, "Recall", "Precision",
           f"Precision-Recall  (mAP@0.5={np.nan_to_num(M['ap50']).mean():.3f})",
           outdir / "PR_curve.png", f"all classes {np.nan_to_num(M['ap50']).mean():.3f} mAP@0.5")
    f1 = M["f1_curve"]; f1m = f1.mean(0); bi = int(f1m.argmax())
    _curve(px, f1, names, "Confidence", "F1",
           f"F1-Confidence  (best {f1m[bi]:.2f} @ {px[bi]:.2f})",
           outdir / "F1_curve.png", f"all classes {f1m[bi]:.2f} @ {px[bi]:.2f}")
    _curve(px, M["p_curve"], names, "Confidence", "Precision", "Precision-Confidence",
           outdir / "P_curve.png", "all classes")
    _curve(px, M["r_curve"], names, "Confidence", "Recall", "Recall-Confidence",
           outdir / "R_curve.png", "all classes")

def plot_labels(gt_cls_all, gt_xywh_norm, names, save):
    """gt_xywh_norm: (M,4) cx,cy,w,h normalized [0,1]."""
    fig, ax = plt.subplots(2, 2, figsize=(12, 10))
    nc = len(names)
    cnt = np.bincount(gt_cls_all.astype(int), minlength=nc)
    ax[0, 0].bar(range(nc), cnt, color="steelblue")
    ax[0, 0].set_xticks(range(nc)); ax[0, 0].set_xticklabels(names, rotation=90, fontsize=7)
    ax[0, 0].set_title("instances per class"); ax[0, 0].set_ylabel("count")
    g = gt_xywh_norm
    if len(g):
        ax[0, 1].hist2d(g[:, 0], g[:, 1], bins=50, cmap="viridis")
        ax[0, 1].set_title("box center  x-y"); ax[0, 1].set_xlabel("x"); ax[0, 1].set_ylabel("y")
        ax[0, 1].set_xlim(0, 1); ax[0, 1].set_ylim(1, 0)
        ax[1, 0].hist2d(g[:, 2], g[:, 3], bins=50, cmap="viridis")
        ax[1, 0].set_title("box  width-height"); ax[1, 0].set_xlabel("width"); ax[1, 0].set_ylabel("height")
        asp = g[:, 2] / np.clip(g[:, 3], 1e-6, None)
        ax[1, 1].hist(np.clip(asp, 0, 5), bins=60, color="steelblue")
        ax[1, 1].set_title("aspect ratio (w/h)")
    fig.tight_layout(); fig.savefig(save, dpi=150); plt.close(fig)

def plot_results(metrics_csv, save):
    """Parse RF-DETR's Lightning metrics.csv → grid of metric-vs-epoch curves."""
    import pandas as pd
    df = pd.read_csv(metrics_csv)
    epoch_col = "epoch" if "epoch" in df.columns else ("step" if "step" in df.columns else None)
    if epoch_col is None:
        epoch_col = "_row"
        df[epoch_col] = np.arange(len(df))
    num = [c for c in df.columns if c not in (epoch_col, "step") and
           np.issubdtype(df[c].dtype, np.number)]
    # collapse to one row per epoch (mean of non-NaN) so train/val logs align
    g = df.groupby(epoch_col)[num].mean(numeric_only=True)
    cols = [c for c in num if g[c].notna().any()]
    if not cols:
        return False
    ncol = min(4, len(cols)); nrow = math.ceil(len(cols) / ncol)
    fig, ax = plt.subplots(nrow, ncol, figsize=(4 * ncol, 3 * nrow), squeeze=False)
    for k, c in enumerate(cols):
        a = ax[k // ncol][k % ncol]
        s = g[c].dropna()
        a.plot(s.index, s.values, marker=".", ms=3, lw=1.2)
        a.set_title(c, fontsize=9); a.set_xlabel(epoch_col); a.grid(alpha=0.3)
    for k in range(len(cols), nrow * ncol):
        ax[k // ncol][k % ncol].axis("off")
    fig.suptitle("Training results", fontsize=13)
    fig.tight_layout(); fig.savefig(save, dpi=150); plt.close(fig)
    return True


In [ ]:
# ============================================================
# DIAGNOSTICS — run predictions on the eval split + collect GT
# ============================================================
import numpy as np, json, math
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

EVAL_SPLIT   = "test"            # "test" or "valid"
PRED_CONF_LOW = 0.05             # low threshold -> full PR curve (filter later)
CM_CONF, CM_IOU = 0.25, 0.45     # confusion-matrix operating point (YOLO defaults)
IOU_VEC = np.linspace(0.5, 0.95, 10)   # COCO IoU sweep for mAP@0.5:0.95

METRICS_DIR = Path(OUTPUT_DIR) / f"metrics_{EVAL_SPLIT}"
METRICS_DIR.mkdir(parents=True, exist_ok=True)
NC = len(CLASS_NAMES)
_name_to_idx = {n: i for i, n in enumerate(CLASS_NAMES)}

# ── ground truth (COCO) ──
_ann = json.load(open(Path(DATASET_DIR) / EVAL_SPLIT / "_annotations.coco.json"))
_imgs_meta = {im["id"]: im for im in _ann["images"]}
_catname   = {c["id"]: c["name"] for c in _ann["categories"]}
_gt = {}
for a in _ann["annotations"]:
    _ci = _name_to_idx.get(_catname.get(a["category_id"]))
    if _ci is None:
        continue
    x, y, w, h = a["bbox"]
    _gt.setdefault(a["image_id"], []).append(([x, y, x + w, y + h], _ci))

# ── predict + collect ──
_images = []; _tp = []; _conf = []; _pcls = []; _tcls = []; _gt_cls_all = []; _gt_xywh = []
for _iid, _im in tqdm(_imgs_meta.items(), desc=f"Predict {EVAL_SPLIT}"):
    _W, _H = _im["width"], _im["height"]
    _pil = Image.open(Path(DATASET_DIR) / EVAL_SPLIT / _im["file_name"]).convert("RGB")
    _d = model.predict(_pil, threshold=PRED_CONF_LOW)
    if len(_d) > 0:
        _px = np.asarray(_d.xyxy, float).reshape(-1, 4)
        _pc = np.asarray(_d.confidence, float)
        _names = _d.data.get("class_name", None)
        if _names is not None and len(_names) == len(_pc):
            _pl = np.array([_name_to_idx.get(str(n), -1) for n in _names])
        else:
            _pl = np.asarray(_d.class_id, int)
            if _pl.size and _pl.max() >= NC:   # 1-indexed COCO fallback
                _pl = _pl - 1
        _k = _pl >= 0
        _px, _pc, _pl = _px[_k], _pc[_k], _pl[_k]
    else:
        _px = np.zeros((0, 4)); _pc = np.zeros(0); _pl = np.zeros(0, int)
    _g = _gt.get(_iid, [])
    _gx = np.array([b for b, _ in _g], float).reshape(-1, 4)
    _gc = np.array([c for _, c in _g], int)
    _images.append(dict(pred_xyxy=_px, pred_conf=_pc, pred_cls=_pl, gt_xyxy=_gx, gt_cls=_gc))
    _tp.append(match_image(box_iou(_px, _gx), _pl, _gc, IOU_VEC))
    _conf.append(_pc); _pcls.append(_pl); _tcls.append(_gc); _gt_cls_all.append(_gc)
    for x1, y1, x2, y2 in _gx:
        _gt_xywh.append([((x1 + x2) / 2) / _W, ((y1 + y2) / 2) / _H, (x2 - x1) / _W, (y2 - y1) / _H])

_tp   = np.concatenate(_tp) if _tp else np.zeros((0, len(IOU_VEC)), bool)
_conf = np.concatenate(_conf) if _conf else np.zeros(0)
_pcls = np.concatenate(_pcls) if _pcls else np.zeros(0, int)
_tcls = np.concatenate(_tcls) if _tcls else np.zeros(0, int)
print(f"Collected {len(_conf)} predictions, {len(_tcls)} GT instances over {len(_images)} images "
      f"({EVAL_SPLIT}).")


In [ ]:
# ============================================================
# DIAGNOSTICS — compute metrics + write all flatbug-style outputs
# ============================================================
import pandas as pd, shutil

_M  = compute_metrics(_tp, _conf, _pcls, _tcls, NC, IOU_VEC)
_cm = confusion_matrix(_images, NC, conf_thres=CM_CONF, iou_thres=CM_IOU)

plot_confusion(_cm, CLASS_NAMES, METRICS_DIR / "confusion_matrix.png", normalize=False)
plot_confusion(_cm, CLASS_NAMES, METRICS_DIR / "confusion_matrix_normalized.png", normalize=True)
plot_curves(_M, CLASS_NAMES, METRICS_DIR)
plot_labels(np.concatenate(_gt_cls_all) if _gt_cls_all else np.zeros(0, int),
            np.array(_gt_xywh).reshape(-1, 4), CLASS_NAMES, METRICS_DIR / "labels.jpg")

# training curves from RF-DETR's Lightning metrics.csv
_mcsv = Path(OUTPUT_DIR) / "metrics.csv"
if _mcsv.exists():
    shutil.copy(_mcsv, METRICS_DIR / "results.csv")
    plot_results(_mcsv, METRICS_DIR / "results.png")
else:
    print(f"(no metrics.csv in {OUTPUT_DIR} - skipping results.png)")

# per-class table (+ mean P/R at best-F1 conf, like the YOLO printout)
_f1m = _M["f1_curve"].mean(0); _bi = int(_f1m.argmax())
# AP@75 per class. _M["ap"] holds all 10 IoU thresholds (0.50..0.95 step 0.05), so
# column 5 IS AP75 — it was being computed and never written. The tiled notebooks
# report per-class AP75, so without this the comparison has a hole on the whole-tray
# row, and AP75 is the metric the two-stage experiment is built around.
_ap75 = _M["ap"][:, 5] if "ap" in _M and _M["ap"].ndim == 2 and _M["ap"].shape[1] > 5 else None
_rows = [dict(**{"class": n}, instances=int(_M["n_gt"][i]), preds=int(_M["n_p"][i]),
              P=round(float(_M["p_curve"][i, _bi]), 4), R=round(float(_M["r_curve"][i, _bi]), 4),
              mAP50=round(float(_M["ap50"][i]), 4),
              mAP75=(round(float(_ap75[i]), 4) if _ap75 is not None else None),
              mAP50_95=round(float(_M["ap5095"][i]), 4))
         for i, n in enumerate(CLASS_NAMES)]
_all = dict(**{"class": "ALL"}, instances=int(_M["n_gt"].sum()), preds=int(_M["n_p"].sum()),
            P=round(float(_M["p_curve"][:, _bi].mean()), 4), R=round(float(_M["r_curve"][:, _bi].mean()), 4),
            mAP50=round(float(np.nan_to_num(_M["ap50"]).mean()), 4),
            mAP75=(round(float(np.nan_to_num(_ap75).mean()), 4) if _ap75 is not None else None),
            mAP50_95=round(float(np.nan_to_num(_M["ap5095"]).mean()), 4))
_pcdf = pd.DataFrame(_rows + [_all])
_pcdf.to_csv(METRICS_DIR / "per_class_metrics.csv", index=False)

print("\n=== EVAL METRICS (" + EVAL_SPLIT + ") ===")
print(f"  mAP50-95 : {_all['mAP50_95']:.4f}")
print(f"  mAP50    : {_all['mAP50']:.4f}")
if _all.get("mAP75") is not None:
    print(f"  mAP75    : {_all['mAP75']:.4f}")
print(f"  P (best-F1 conf {_M['px'][_bi]:.2f}) : {_all['P']:.4f}")
print(f"  R (best-F1 conf {_M['px'][_bi]:.2f}) : {_all['R']:.4f}")
print(_pcdf.to_string(index=False))
print(f"\nAll plots + CSVs saved to: {METRICS_DIR}")
for _f in sorted(METRICS_DIR.glob("*")):
    print("   ", _f.name)


In [ ]:
# ============================================================
# DIAGNOSTICS — val_batch previews (GT labels vs predictions)
# ============================================================
import random as _rnd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

_sample = _rnd.Random(0).sample(list(_imgs_meta), min(9, len(_imgs_meta)))
_palette = plt.cm.tab20(np.linspace(0, 1, NC))

for _tag, _use_pred in [("labels", False), ("pred", True)]:
    _c = 3; _r = math.ceil(len(_sample) / _c)
    fig, axes = plt.subplots(_r, _c, figsize=(5 * _c, 4 * _r))
    axes = np.array(axes).reshape(-1)
    for _ax, _iid in zip(axes, _sample):
        _im = _imgs_meta[_iid]
        _pil = Image.open(Path(DATASET_DIR) / EVAL_SPLIT / _im["file_name"]).convert("RGB")
        _ax.imshow(_pil); _ax.axis("off"); _ax.set_title(_im["file_name"][:24], fontsize=7)
        if _use_pred:
            _d = model.predict(_pil, threshold=CM_CONF)
            _nm = _d.data.get("class_name", []) if len(_d) > 0 else []
            for _bi2, ((x1, y1, x2, y2), cf) in enumerate(
                    zip(_d.xyxy, _d.confidence) if len(_d) > 0 else []):
                _ci = _name_to_idx.get(str(_nm[_bi2]), 0) if _bi2 < len(_nm) else 0
                _ax.add_patch(mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1, lw=1,
                              edgecolor=_palette[_ci % NC], facecolor="none"))
        else:
            for (x1, y1, x2, y2), _ci in _gt.get(_iid, []):
                _ax.add_patch(mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1, lw=1,
                              edgecolor=_palette[_ci % NC], facecolor="none"))
    for _ax in axes[len(_sample):]:
        _ax.axis("off")
    fig.suptitle(f"{EVAL_SPLIT}_batch_{_tag}", fontsize=13)
    fig.tight_layout()
    fig.savefig(METRICS_DIR / f"val_batch_{_tag}.jpg", dpi=130, bbox_inches="tight")
    plt.show()
print(f"Saved val_batch_labels.jpg / val_batch_pred.jpg -> {METRICS_DIR}")


---
## Cell 10 — Export to ONNX

Exports the trained model to ONNX format for deployment (e.g. in Python, C++, or on-device with ONNX Runtime).

**Requires:** `pip install "rf-detr-1.7.1/[onnx]"` (run once).


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# EXPORT SETTINGS  ← change these if needed
# ─────────────────────────────────────────────────────────────────────────────

# Folder where the .onnx file will be written.
EXPORT_DIR = str(Path(OUTPUT_DIR) / "export")

# ONNX opset version.  17 is a safe modern default.
OPSET_VERSION = 17

# Static batch size baked into the ONNX graph.  1 = single-image inference.
# Set dynamic_batch=True to support variable batch sizes at runtime.
EXPORT_BATCH_SIZE = 1
DYNAMIC_BATCH     = False

# ─────────────────────────────────────────────────────────────────────────────

# Install ONNX extras if not already present.
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", "rf-detr-1.7.1/[onnx]", "--quiet"],
    check=True
)

onnx_path = model.export(
    output_dir     = EXPORT_DIR,
    format         = "onnx",
    opset_version  = OPSET_VERSION,
    batch_size     = EXPORT_BATCH_SIZE,
    dynamic_batch  = DYNAMIC_BATCH,
    verbose        = True,
)

print(f"\nONNX model exported to: {onnx_path}")


---
## Cell 11 — Resume Training (Optional)

If training was interrupted, or you want to continue from a saved checkpoint with more epochs, run this cell instead of Cell 6.

Just update `RESUME_FROM` in Cell 2 to the path of the checkpoint you want to continue from,
and increase `EPOCHS` to the total number of epochs you want to reach.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Override these locally without touching Cell 2
# ─────────────────────────────────────────────────────────────────────────────

# Checkpoint to resume from.  Usually the last-saved epoch or the best checkpoint.
_resume_path = str(Path(OUTPUT_DIR) / "checkpoint_last.pth")

# Total epochs to train up to (not additional epochs — RF-DETR reads the saved
# epoch number and continues from there, so set this to your final target).
_resume_epochs = EPOCHS + 50  # example: add 50 more epochs

# ─────────────────────────────────────────────────────────────────────────────

from pathlib import Path as _Path
if not _Path(_resume_path).exists():
    raise FileNotFoundError(f"Checkpoint not found: {_resume_path}")

model.train(
    dataset_dir         = DATASET_DIR,
    output_dir          = OUTPUT_DIR,
    class_names         = CLASS_NAMES,
    epochs              = _resume_epochs,
    batch_size          = BATCH_SIZE,
    grad_accum_steps    = GRAD_ACCUM_STEPS,
    lr                  = LR,
    lr_encoder          = LR_ENCODER,
    lr_drop             = LR_DROP,
    warmup_epochs       = WARMUP_EPOCHS,
    weight_decay        = WEIGHT_DECAY,
    multi_scale         = MULTI_SCALE,
    use_ema             = USE_EMA,
    checkpoint_interval = CHECKPOINT_INTERVAL,
    eval_interval       = EVAL_INTERVAL,
    run_test            = RUN_TEST,
    early_stopping      = EARLY_STOPPING,
    early_stopping_patience  = EARLY_STOPPING_PATIENCE,
    early_stopping_min_delta = EARLY_STOPPING_MIN_DELTA,
    tensorboard         = TENSORBOARD,
    wandb               = WANDB,
    num_workers         = NUM_WORKERS,
    resume              = _resume_path,   # ← resumes from checkpoint
)
